# haan: telephony fine-tune of Smart Turn v3.2 (Colab T4)

Two arms with the same init (Daily's v3.2 weights recovered from ONNX), shards, seed, steps and learning rate:
- `control`: wideband only
- `telephony`: each clip randomly wideband / 8 kHz mu-law / PSTN voice band

Each arm is then scored on the full official test set (31,527 clips) under all three channels.
Runtime → Change runtime type → T4 GPU. About 1.5 hours in total. Outputs are copied to Google Drive so a disconnect loses nothing.

In [ ]:
# Settings
ARMS = ['control', 'telephony']
SHARDS = 12        # of 83 training shards (~3.3k clips each)
EVERY = 1          # score every k-th test clip (1 = full test set)
DRIVE_DIR = '/content/drive/MyDrive/haan'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p {DRIVE_DIR}
!git clone -q https://github.com/Charansripadi/haan /content/haan
%cd /content/haan
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
hf_hub_download('pipecat-ai/smart-turn-v3', 'smart-turn-v3.2-gpu.onnx', local_dir='models')
snapshot_download('pipecat-ai/smart-turn-data-v3.2-test', repo_type='dataset', local_dir='data/st-v3.2-test', max_workers=8)
!python -m pytest -q

In [ ]:
import os
for arm in ARMS:
    out = f'{DRIVE_DIR}/runs/{arm}'
    if os.path.exists(f'{out}/model.pt'):
        print('skip training', arm); continue
    !python scripts/train.py --arm {arm} --shards {SHARDS} --batch 64 --workers 2 --out {out}

In [ ]:
for arm in ARMS:
    !python scripts/bench_telephony.py --torch-weights {DRIVE_DIR}/runs/{arm}/model.pt --every {EVERY} --batch 128 --out {DRIVE_DIR}/results/scores_{arm}

In [ ]:
for arm in ARMS:
    !python scripts/report_telephony.py --scores {DRIVE_DIR}/results/scores_{arm} --title {arm} --out {DRIVE_DIR}/results/report_{arm}.md
    print()

In [ ]:
# Score Daily's original v3.2 (FP32 weights recovered from ONNX) on the same full test set,
# so the README can compare original vs control vs telephony like-for-like.
import os
os.makedirs(f'{DRIVE_DIR}/runs/original', exist_ok=True)
!python -m haan.convert models/smart-turn-v3.2-gpu.onnx {DRIVE_DIR}/runs/original/model.pt
!python scripts/bench_telephony.py --torch-weights {DRIVE_DIR}/runs/original/model.pt --every {EVERY} --batch 128 --out {DRIVE_DIR}/results/scores_original
!python scripts/report_telephony.py --scores {DRIVE_DIR}/results/scores_original --title original --out {DRIVE_DIR}/results/report_original.md